# Module 5: Executive Narrative and Simulation Dashboard

This notebook implements README Module 5 and turns model outputs into a decision narrative for business users.

Implementation rule alignment: transformation and reporting logic is imported from `src/module5_reporting.py`, while this notebook focuses on orchestration, diagnostics, and visuals.

Covered here:
- John Smith style case study for one household,
- utility decomposition (why each item won),
- ablation impact proof (Variant 0 -> Variant 3),
- recommendation simulator export for dashboard usage.

In [4]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_RAW = ROOT / 'data' / '01_raw'
DATA_PROCESSED = ROOT / 'data' / '02_processed'
REPORTS = ROOT / 'reports'
FIGURES = REPORTS / 'figures'

FIGURES.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

In [5]:
from src.data_loader import load_or_build_master_transactions
from src.module5_reporting import (
    build_case_study,
    build_recommendation_simulator_table,
    build_ablation_proof_table,
)
# Avoid cached parquet schema issues in master transaction artifacts.
master_tx_dir = DATA_PROCESSED
all_path = master_tx_dir / 'master_transactions_all.parquet'
organic_path = master_tx_dir / 'master_transactions_organic_only.parquet'
if all_path.exists() or organic_path.exists():
    master_tx_dir = master_tx_dir / 'notebook_tmp'
    master_tx_dir.mkdir(parents=True, exist_ok=True)

tx_all, tx_organic = load_or_build_master_transactions(DATA_RAW, master_tx_dir)
hh_demographic = pd.read_csv(DATA_RAW / 'hh_demographic.csv')
scored_candidates = pd.read_csv(DATA_PROCESSED / 'candidate_set_module3_scored.csv')
top5 = pd.read_csv(DATA_PROCESSED / 'top5_recommendations_module3.csv')
ablation_summary = pd.read_csv(DATA_PROCESSED / 'module4_ablation_summary.csv')

print('tx_all rows:', len(tx_all))
print('scored_candidates rows:', len(scored_candidates))
print('top5 rows:', len(top5))
print('ablation summary rows:', len(ablation_summary))
display(top5.head(5))

tx_all rows: 2595732
scored_candidates rows: 23544
top5 rows: 2500
ablation summary rows: 4


,household_key,COMMODITY_DESC,seed_items,relevance_als,relevance_mba,Relevance,source,source_detail,snapshot_week,was_recently_purchased,...,Normalized_Margin,deal_sensitivity,is_active_campaign_period,item_is_promoted,Context,Utility,Relevance_Contribution,Uplift_Contribution,Margin_Contribution,Context_Contribution
0,1,POPCORN,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.947021,0.000000,0.947021,als,ALS,102,False,...,1.0,0.813953,False,True,0.5,0.900901,0.378808,0.247093,0.2,0.075
1,1,VEGETABLES - ALL OTHERS,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.000000,1.000000,1.000000,mba,MBA,102,False,...,1.0,0.813953,False,True,0.5,0.898837,0.400000,0.223837,0.2,0.075
2,1,BEEF,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.000000,0.796527,0.796527,mba,MBA,102,False,...,1.0,0.813953,False,True,0.5,0.837797,0.318611,0.244186,0.2,0.075
3,1,BLEACH,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.721154,0.000000,0.721154,als,ALS,102,False,...,1.0,0.813953,False,True,0.5,0.798927,0.288462,0.235465,0.2,0.075
4,1,CHRISTMAS SEASONAL,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.523417,0.000000,0.523417,als,ALS,102,False,...,1.0,0.813953,False,True,0.5,0.728553,0.209367,0.244186,0.2,0.075


In [6]:
case = build_case_study(
    history=tx_all,
    scored_candidates=scored_candidates,
    top5=top5,
    household_key=None,
    top_k=5,
)

print('Selected case-study household:', case.household_key)
display(case.comparison_table)
display(case.component_decomposition)

Selected case-study household: 149


,Chimera_Item,Chimera_Utility,Variant0_Item,Variant0_Relevance
0,DOG FOODS,1.000000,DOG FOODS,1.000000
1,DRIED FRUIT,0.995349,DRIED FRUIT,1.000000
2,PICKLE/RELISH/PKLD VEG,0.886138,COUPONS/STORE & MFG,1.000000
3,SOAP - LIQUID & BAR,0.862889,PICKLE/RELISH/PKLD VEG,0.732787
4,HAND/BODY/FACIAL PRODUCTS,0.811174,SOAP - LIQUID & BAR,0.683386


,COMMODITY_DESC,Relevance,Uplift,Normalized_Margin,Context,Utility,Relevance_Contribution,Uplift_Contribution,Margin_Contribution,Context_Contribution
740,DOG FOODS,1.000000,1.000000,1.0,1.0,1.000000,0.400000,0.250000,0.2,0.15
741,DRIED FRUIT,1.000000,0.981395,1.0,1.0,0.995349,0.400000,0.245349,0.2,0.15
742,PICKLE/RELISH/PKLD VEG,0.732787,0.972093,1.0,1.0,0.886138,0.293115,0.243023,0.2,0.15
743,SOAP - LIQUID & BAR,0.683386,0.958140,1.0,1.0,0.862889,0.273355,0.239535,0.2,0.15
744,HAND/BODY/FACIAL PRODUCTS,0.539563,0.981395,1.0,1.0,0.811174,0.215825,0.245349,0.2,0.15


In [7]:
timeline = case.history_timeline.copy()

fig_timeline = go.Figure()
fig_timeline.add_trace(
    go.Scatter(
        x=timeline['DAY'],
        y=timeline['revenue'],
        mode='lines+markers',
        name='Revenue',
        line=dict(width=2),
    )
)
fig_timeline.add_trace(
    go.Bar(
        x=timeline['DAY'],
        y=timeline['promoted_rows'],
        name='Promoted Rows',
        opacity=0.35,
        yaxis='y2',
    )
)
fig_timeline.update_layout(
    title=f'Case Study Household {case.household_key}: Revenue Timeline with Promo Intensity',
    xaxis_title='Day',
    yaxis_title='Revenue_Retailer',
    yaxis2=dict(title='Promoted Rows', overlaying='y', side='right', showgrid=False),
    height=460,
    barmode='overlay',
)
fig_timeline.show()

decomp_long = case.component_decomposition.melt(
    id_vars=['COMMODITY_DESC', 'Utility'],
    value_vars=[
        'Relevance_Contribution',
        'Uplift_Contribution',
        'Margin_Contribution',
        'Context_Contribution',
    ],
    var_name='Component',
    value_name='Contribution',
)
fig_decomp = px.bar(
    decomp_long,
    x='Contribution',
    y='COMMODITY_DESC',
    color='Component',
    orientation='h',
    barmode='stack',
    title=f'Utility Decomposition for Household {case.household_key}',
)
fig_decomp.update_layout(height=460, yaxis_title='Commodity')
fig_decomp.show()

In [8]:
proof_table = build_ablation_proof_table(ablation_summary)
display(proof_table)

fig_proof = px.bar(
    proof_table,
    x='Variant',
    y=['Incremental_Precision@5', 'Average_Recommended_Margin'],
    barmode='group',
    title='Ablation Proof Screen: Variant Performance',
)
fig_proof.update_layout(height=460, xaxis_title='', yaxis_title='Score')
fig_proof.show()

,Variant,Incremental_Precision@5,Average_Recommended_Margin,Average_Incremental_Hits,Average_Targets,Precision_Lift_vs_Baseline,Margin_Lift_vs_Baseline,Margin_Lift_vs_Popularity,Precision_Lift_%,Margin_Lift_%,Margin_Lift_vs_Popularity_%
0,Variant 0 - Relevance only,0.064301,0.886848,0.321503,11.609603,0.000000,0.000000,0.007135,0.00,0.00,0.71
1,Variant 1 - Relevance + Uplift,0.065971,0.854697,0.329854,11.609603,0.025974,-0.036252,-0.029376,2.60,-3.63,-2.94
2,Variant 2 - Relevance + Uplift + Margin,0.067641,0.963674,0.338205,11.609603,0.051948,0.086629,0.094382,5.19,8.66,9.44
3,Variant 3 - Full Chimera,0.069311,0.985386,0.346555,11.609603,0.077922,0.111111,0.119039,7.79,11.11,11.90


In [9]:
simulator_table = build_recommendation_simulator_table(
    hh_demographic=hh_demographic,
    top5=top5,
)

simulator_path = DATA_PROCESSED / 'module5_recommendation_simulator.csv'
case_path = DATA_PROCESSED / 'module5_case_study_comparison.csv'

simulator_table.to_csv(simulator_path, index=False)
case.comparison_table.to_csv(case_path, index=False)

fig_timeline.write_html(FIGURES / 'john_smith_case_study.html')
fig_decomp.write_html(FIGURES / 'utility_decomposition.html')
fig_proof.write_html(FIGURES / 'ablation_proof_screen.html')

print('saved:', simulator_path)
print('saved:', case_path)
print('saved:', FIGURES / 'john_smith_case_study.html')
print('saved:', FIGURES / 'utility_decomposition.html')
print('saved:', FIGURES / 'ablation_proof_screen.html')

display(simulator_table.head(10))

saved: C:\Users\dell\Downloads\New folder (9)\Data_Driven_Marketing_complete-journey\data\02_processed\module5_recommendation_simulator.csv
saved: C:\Users\dell\Downloads\New folder (9)\Data_Driven_Marketing_complete-journey\data\02_processed\module5_case_study_comparison.csv
saved: C:\Users\dell\Downloads\New folder (9)\Data_Driven_Marketing_complete-journey\reports\figures\john_smith_case_study.html
saved: C:\Users\dell\Downloads\New folder (9)\Data_Driven_Marketing_complete-journey\reports\figures\utility_decomposition.html
saved: C:\Users\dell\Downloads\New folder (9)\Data_Driven_Marketing_complete-journey\reports\figures\ablation_proof_screen.html


,household_key,COMMODITY_DESC,seed_items,relevance_als,relevance_mba,Relevance,source,source_detail,snapshot_week,was_recently_purchased,...,Context,Utility,Relevance_Contribution,Uplift_Contribution,Margin_Contribution,Context_Contribution,HOMEOWNER_DESC,KID_CATEGORY_DESC,Utility_Formula_Text,Utility_Expanded_Text
0,1,POPCORN,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.947021,0.000000,0.947021,als,ALS,102,False,...,0.5,0.900901,0.378808,0.247093,0.2,0.075,Homeowner,None/Unknown,0.40*R + 0.25*U + 0.20*M + 0.15*C,0.379 + 0.247 + 0.2 + 0.075 = 0.901
1,1,VEGETABLES - ALL OTHERS,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.000000,1.000000,1.000000,mba,MBA,102,False,...,0.5,0.898837,0.400000,0.223837,0.2,0.075,Homeowner,None/Unknown,0.40*R + 0.25*U + 0.20*M + 0.15*C,0.4 + 0.224 + 0.2 + 0.075 = 0.899
2,1,BEEF,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.000000,0.796527,0.796527,mba,MBA,102,False,...,0.5,0.837797,0.318611,0.244186,0.2,0.075,Homeowner,None/Unknown,0.40*R + 0.25*U + 0.20*M + 0.15*C,0.319 + 0.244 + 0.2 + 0.075 = 0.838
3,1,BLEACH,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.721154,0.000000,0.721154,als,ALS,102,False,...,0.5,0.798927,0.288462,0.235465,0.2,0.075,Homeowner,None/Unknown,0.40*R + 0.25*U + 0.20*M + 0.15*C,0.288 + 0.235 + 0.2 + 0.075 = 0.799
4,1,CHRISTMAS SEASONAL,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.523417,0.000000,0.523417,als,ALS,102,False,...,0.5,0.728553,0.209367,0.244186,0.2,0.075,Homeowner,None/Unknown,0.40*R + 0.25*U + 0.20*M + 0.15*C,0.209 + 0.244 + 0.2 + 0.075 = 0.729
5,2,SANDWICHES,APPLES | BAG SNACKS | BAKED BREAD/BUNS/ROLLS,0.000000,1.000000,1.000000,mba,MBA,102,False,...,0.5,0.925000,0.400000,0.250000,0.2,0.075,NaN,NaN,0.40*R + 0.25*U + 0.20*M + 0.15*C,0.4 + 0.25 + 0.2 + 0.075 = 0.925
6,2,CITRUS,APPLES | BAG SNACKS | BAKED BREAD/BUNS/ROLLS,0.291810,1.000000,1.000000,both,BOTH,102,False,...,0.5,0.913889,0.400000,0.238889,0.2,0.075,NaN,NaN,0.40*R + 0.25*U + 0.20*M + 0.15*C,0.4 + 0.239 + 0.2 + 0.075 = 0.914
7,2,LUNCHMEAT,APPLES | BAG SNACKS | BAKED BREAD/BUNS/ROLLS,0.000000,1.000000,1.000000,mba,MBA,102,False,...,0.5,0.913889,0.400000,0.238889,0.2,0.075,NaN,NaN,0.40*R + 0.25*U + 0.20*M + 0.15*C,0.4 + 0.239 + 0.2 + 0.075 = 0.914
8,2,DELI MEATS,APPLES | BAG SNACKS | BAKED BREAD/BUNS/ROLLS,0.000000,1.000000,1.000000,mba,MBA,102,False,...,0.5,0.908333,0.400000,0.233333,0.2,0.075,NaN,NaN,0.40*R + 0.25*U + 0.20*M + 0.15*C,0.4 + 0.233 + 0.2 + 0.075 = 0.908
9,2,TOMATOES,APPLES | BAG SNACKS | BAKED BREAD/BUNS/ROLLS,0.000000,1.000000,1.000000,mba,MBA,102,False,...,0.5,0.897222,0.400000,0.222222,0.2,0.075,NaN,NaN,0.40*R + 0.25*U + 0.20*M + 0.15*C,0.4 + 0.222 + 0.2 + 0.075 = 0.897


In [10]:
required_cols = [
    'household_key',
    'COMMODITY_DESC',
    'Utility',
    'Utility_Formula_Text',
    'Utility_Expanded_Text',
]
missing_cols = [c for c in required_cols if c not in simulator_table.columns]

if missing_cols:
    raise ValueError(f'Module 5 simulator is missing columns: {missing_cols}')
if simulator_table.empty:
    raise ValueError('Module 5 simulator table is empty.')
if case.comparison_table.empty:
    raise ValueError('Module 5 case-study comparison table is empty.')

print('Module 5 completed successfully.')
print('Simulator rows:', len(simulator_table))
print('Case-study household:', case.household_key)

Module 5 completed successfully.
Simulator rows: 2500
Case-study household: 149
